[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/halla-ai/deepnlp-2026/blob/main/notebooks/week-03.ipynb)

# 3주차 실습: 제주 공개 문서의 LoRA 미세조정

- 제공된 학습·평가 함수를 실행, 랭크 선택 한 곳만 변경하는 골격
- 기본 비교: r = 4와 r = 8을 각각 새 모델에서 실제 학습·평가
- 한국어 공개 데이터: 위키백과 「제주특별자치도」 고정 판 42464087
- 과제: 보이는 텍스트에 자연·지형 키워드가 포함됐는지의 교수 규칙 기반 proxy 분류
- 감성 분류와 사람의 주제 gold가 아님. 규칙을 얼마나 모방했는지의 작은 동작 확인
- CPU, 최대 입력 64토큰, 각 설정 6 step, 배치 2. 데이터 8개 학습·4개 평가
- 전체 미세조정은 파라미터 수만 계산. 전체 학습의 시간·정확도·peak 메모리 비교는 미실행


## 0. 준비

- 모델과 공개 원문은 실행 때 다운로드, 실패를 가짜 데이터로 대체하지 않는 원칙
- torch CPU 스레드 2, Trainer도 CPU 고정. GPU 의존 없음


In [1]:
%pip -q install transformers datasets peft accelerate


## 1. 제주 원문과 한국어 tokenizer

- 원문: https://ko.wikipedia.org/w/index.php?oldid=42464087
- 저자: 해당 판의 역사에 등재된 위키백과 기여자, CC BY-SA 4.0
- 원문 문단에서 최대 52토큰 조각을 추출, 특별 토큰을 포함해 최대 64토큰 입력
- proxy label 1: 보이는 조각에 기후·지형·화산·해안·한라산 중 하나 포함
- proxy label 0: 해당 문자열 없음. 실제 주제의 부정이나 감정의 부정을 의미하지 않음
- 동일 원문 문단이 학습·평가에 동시에 들어가지 않도록 문단 ID 기준 분리


In [2]:
import gc, html, json, random, re, time, urllib.parse, urllib.request
from pathlib import Path
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, set_seed
from peft import LoraConfig, get_peft_model

torch.set_num_threads(2)
SEED = 42
MODEL_NAME = "monologg/koelectra-base-v3-discriminator"
MAX_LENGTH = 64
STEPS = 6
BATCH_SIZE = 2
OLDID = 42464087
SOURCE = f"https://ko.wikipedia.org/w/index.php?oldid={OLDID}"
TERMS = ("기후", "지형", "화산", "해안", "한라산")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
url = "https://ko.wikipedia.org/w/api.php?" + urllib.parse.urlencode({
    "action": "parse", "oldid": OLDID, "prop": "text", "format": "json", "formatversion": 2})
request = urllib.request.Request(url, headers={"User-Agent": "deepnlp-course/1.0 (yj.lee@chu.ac.kr)"})
with urllib.request.urlopen(request, timeout=60) as response:
    parsed = json.load(response)["parse"]
assert parsed["revid"] == OLDID, "수집 판 번호 불일치"

def clean(fragment):
    fragment = re.sub(r"<sup\b[^>]*>.*?</sup>", "", fragment, flags=re.S)
    return re.sub(r"\s+", " ", html.unescape(re.sub(r"<[^>]+>", " ", fragment))).strip()

paragraphs = [clean(p) for p in re.findall(r"<p\b[^>]*>(.*?)</p>", parsed["text"], flags=re.S)]
paragraphs = [p for p in paragraphs if len(p) >= 40]
examples = []
for pid, paragraph in enumerate(paragraphs):
    ids = tokenizer(paragraph, add_special_tokens=False)["input_ids"]
    for start in range(0, len(ids), 52):
        window = ids[start:start + 52]
        if len(window) < 8:
            continue
        # 공개 문자열 API로 재토큰화, 4.x/5.x 호환. label은 실제 모델 입력 기준
        visible_text = tokenizer.decode(window, skip_special_tokens=True)
        encoded = tokenizer(visible_text, add_special_tokens=True, padding="max_length",
                            truncation=True, max_length=MAX_LENGTH)
        visible_text = tokenizer.decode(encoded["input_ids"], skip_special_tokens=True)
        label = int(any(term in visible_text for term in TERMS))
        examples.append({"paragraph_id": f"P{pid:03d}", "chunk_id": f"P{pid:03d}-T{start}",
                         "visible_text": visible_text, "labels": label,
                         "input_ids": encoded["input_ids"], "attention_mask": encoded["attention_mask"]})
print("공개 출처:", SOURCE, "라이선스: CC BY-SA 4.0")
print("원문 문단:", len(paragraphs), "사용 가능 조각:", len(examples))
print("규칙 기반 label:", TERMS, "포함=1 / 없음=0")


공개 출처: https://ko.wikipedia.org/w/index.php?oldid=42464087 라이선스: CC BY-SA 4.0
원문 문단: 58 사용 가능 조각: 127
규칙 기반 label: ('기후', '지형', '화산', '해안', '한라산') 포함=1 / 없음=0


## 2. 작은 균형 학습·평가셋

- 각 label에서 서로 다른 문단 6개 선택: 4개 학습, 2개 평가
- 한 문단에서 한 조각만 사용, 모든 설정에서 같은 분할 사용
- 평가 4건이므로 한 건 차이는 정확도 0.25 차이. 일반 성능 추정에 부족한 규모


In [3]:
rng = random.Random(SEED)
shuffled = examples.copy()
rng.shuffle(shuffled)
used_paragraphs, train_rows, eval_rows = set(), [], []
# 키워드 포함 문단을 먼저 확보, 두 클래스 사이에서도 원문 문단 중복 금지
for label in (1, 0):
    chosen = []
    for row in shuffled:
        if row["labels"] == label and row["paragraph_id"] not in used_paragraphs:
            chosen.append(row)
            used_paragraphs.add(row["paragraph_id"])
            if len(chosen) == 6:
                break
    assert len(chosen) == 6, f"label {label}의 독립 문단 부족: 원문·tokenizer·규칙 확인"
    train_rows.extend(chosen[:4])
    eval_rows.extend(chosen[4:])
assert not {r["paragraph_id"] for r in train_rows}.intersection(r["paragraph_id"] for r in eval_rows)
rng.shuffle(train_rows)
rng.shuffle(eval_rows)
columns = ("input_ids", "attention_mask", "labels")
train_ds = Dataset.from_list([{k: row[k] for k in columns} for row in train_rows])
eval_ds = Dataset.from_list([{k: row[k] for k in columns} for row in eval_rows])
print("학습:", len(train_ds), "평가:", len(eval_ds))
print("평가 문단:", [r["paragraph_id"] for r in eval_rows])
for row in eval_rows:
    print(row["chunk_id"], "proxy label", row["labels"], row["visible_text"][:100])


학습: 8 평가: 4
평가 문단: ['P016', 'P026', 'P006', 'P039']
P016-T52 proxy label 1 # # 화산 ) 이 위치해 있다.
P026-T104 proxy label 0 보호조치 등은 경찰관직무집행법에 따라 수행한다. 단, 수사권은 치안유지를 위해 중앙정부 소속의 제주경찰청 소속의 경찰이 수행한다.
P006-T52 proxy label 0 498년 에 백제 동성왕 에 의해 복속되었다. 662년 에는 신라 의 복속이 되었다.
P039-T0 proxy label 1 경사진 지형이 많은 관계로, 경사진 곳에 있는 과수원에서 이용할 목적으로 세레스 같은 4륜구동 트럭을 농가에서 많이 이용한다. 이 때문에 세레스 는 다른 농촌 지역보다 제주도에


## 3. 제공된 LoRA trainer

- 설정마다 같은 seed로 한국어 사전학습 모델과 분류 head를 새로 초기화
- query·value에 LoRA, 분류 head도 학습. rank 외 조건은 동일
- alpha = 2 x rank로 alpha/r = 2 유지. 학생이 바꾸는 값은 rank 한 곳
- 초기/학습 후 평가, 6 step 학습 시간, 파라미터 수와 텐서 bytes 출력
- 텐서 bytes는 파라미터 저장량. 활성값·gradient·optimizer 상태·peak RSS는 미측정
- 시간은 trainer.train() 범위, 다운로드·모델 로딩·평가는 제외


In [4]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"proxy_accuracy": float(np.mean(predictions == labels))}

def run_lora(rank):
    set_seed(SEED)
    base = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, torch_dtype=torch.float32).to("cpu")
    full_count = sum(p.numel() for p in base.parameters())
    full_bytes = sum(p.numel() * p.element_size() for p in base.parameters())
    config = LoraConfig(r=rank, lora_alpha=2 * rank, lora_dropout=0.1, bias="none",
                        task_type="SEQ_CLS", target_modules=["query", "value"])
    adapted = get_peft_model(base, config)
    trainable = sum(p.numel() for p in adapted.parameters() if p.requires_grad)
    adapter_count = sum(p.numel() for name, p in adapted.named_parameters() if p.requires_grad and "lora_" in name)
    head_count = trainable - adapter_count
    parameter_bytes = sum(p.numel() * p.element_size() for p in adapted.parameters())
    trainable_bytes = sum(p.numel() * p.element_size() for p in adapted.parameters() if p.requires_grad)
    args = TrainingArguments(output_dir=f"./week03-r{rank}", use_cpu=True, max_steps=STEPS,
                             per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
                             learning_rate=5e-4, seed=SEED, data_seed=SEED, report_to=[],
                             save_strategy="no", logging_strategy="no", eval_strategy="no",
                             dataloader_num_workers=0, dataloader_pin_memory=False, label_names=["labels"], disable_tqdm=True)
    trainer = Trainer(model=adapted, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
                      compute_metrics=compute_metrics)
    before = trainer.evaluate()
    start = time.perf_counter()
    trainer.train()
    train_seconds = time.perf_counter() - start
    after = trainer.evaluate()
    prediction = trainer.predict(eval_ds).predictions.argmax(axis=-1).tolist()
    result = {"rank": rank, "alpha": 2 * rank, "steps": int(trainer.state.global_step),
              "full_model_parameters_count_only": full_count, "full_model_parameter_bytes_only": full_bytes,
              "trainable_parameters": trainable, "adapter_parameters": adapter_count,
              "head_parameters": head_count, "trainable_parameter_bytes": trainable_bytes,
              "all_parameter_bytes": parameter_bytes, "train_seconds": train_seconds,
              "proxy_accuracy_before": before["eval_proxy_accuracy"],
              "proxy_accuracy_after": after["eval_proxy_accuracy"], "eval_predictions": prediction,
              "full_finetuning_training": "미측정", "peak_memory": "미측정"}
    print(json.dumps(result, ensure_ascii=False, indent=2))
    del trainer, adapted, base
    gc.collect()
    return result


## 4. 먼저 r = 4 기준선 실제 학습·평가

- 모델 파라미터를 세는 것만으로 학습했다고 말하지 않는 원칙
- 아래 함수에서 실제 trainer.train()과 평가 실행


In [5]:
baseline = run_lora(4)


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at monologg/koelectra-base-v3-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


max_steps is given, it will override any value given in num_train_epochs


{'eval_loss': 0.6894850730895996, 'eval_model_preparation_time': 0.006, 'eval_proxy_accuracy': 0.5, 'eval_runtime': 1.1687, 'eval_samples_per_second': 3.423, 'eval_steps_per_second': 1.711}


{'train_runtime': 2.0242, 'train_samples_per_second': 5.928, 'train_steps_per_second': 2.964, 'train_loss': 0.9745088418324789, 'epoch': 1.5}


{'eval_loss': 0.7965513467788696, 'eval_model_preparation_time': 0.006, 'eval_proxy_accuracy': 0.5, 'eval_runtime': 0.3271, 'eval_samples_per_second': 12.23, 'eval_steps_per_second': 6.115, 'epoch': 1.5}


{
  "rank": 4,
  "alpha": 8,
  "steps": 6,
  "full_model_parameters_count_only": 112922882,
  "full_model_parameter_bytes_only": 451691528,
  "trainable_parameters": 739586,
  "adapter_parameters": 147456,
  "head_parameters": 592130,
  "trainable_parameter_bytes": 2958344,
  "all_parameter_bytes": 454649872,
  "train_seconds": 2.2418278999975882,
  "proxy_accuracy_before": 0.5,
  "proxy_accuracy_after": 0.5,
  "eval_predictions": [
    0,
    0,
    0,
    0
  ],
  "full_finetuning_training": "미측정",
  "peak_memory": "미측정"
}


## 5. 랭크 한 곳 선택하고 실제 재학습

- 아래 값 한 곳만 변경. 기본 r = 8도 같은 함수로 실제 학습·평가
- 모든 셀 기본 실행으로 r = 4와 r = 8 비교 완료
- rank를 다른 값으로 바꾸고 이 셀부터 다시 실행해 비교


In [6]:
# TODO: 비교 랭크 하나만 변경 (기본 8, 예: 4 또는 16)
rank = 8


In [7]:
changed = run_lora(rank)
print("\n설정 | 학습 파라미터 | proxy 정확도 전→후 | 학습 초 | 파라미터 MiB")
for result in (baseline, changed):
    print(f"r={result['rank']} | {result['trainable_parameters']:,} | "
          f"{result['proxy_accuracy_before']:.2f}→{result['proxy_accuracy_after']:.2f} | "
          f"{result['train_seconds']:.2f} | {result['all_parameter_bytes'] / 2**20:.2f}")
print("전체 미세조정: 파라미터 count만 계산, 학습 시간·정확도 미측정")
report = {"source": SOURCE, "revision": OLDID, "license": "CC BY-SA 4.0", "label_type": "keyword proxy",
          "label_terms": list(TERMS), "model": MODEL_NAME, "seed": SEED, "max_length": MAX_LENGTH,
          "train_chunks": [r["chunk_id"] for r in train_rows], "eval_chunks": [r["chunk_id"] for r in eval_rows],
          "eval_labels": [r["labels"] for r in eval_rows], "baseline": baseline, "changed": changed}
Path("week03-results.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("실행 결과: week03-results.json")


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at monologg/koelectra-base-v3-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


max_steps is given, it will override any value given in num_train_epochs


{'eval_loss': 0.6894850730895996, 'eval_model_preparation_time': 0.004, 'eval_proxy_accuracy': 0.5, 'eval_runtime': 0.3193, 'eval_samples_per_second': 12.527, 'eval_steps_per_second': 6.263}


{'train_runtime': 2.0849, 'train_samples_per_second': 5.756, 'train_steps_per_second': 2.878, 'train_loss': 0.9750036398569742, 'epoch': 1.5}


{'eval_loss': 0.7982207536697388, 'eval_model_preparation_time': 0.004, 'eval_proxy_accuracy': 0.5, 'eval_runtime': 0.3019, 'eval_samples_per_second': 13.25, 'eval_steps_per_second': 6.625, 'epoch': 1.5}


{
  "rank": 8,
  "alpha": 16,
  "steps": 6,
  "full_model_parameters_count_only": 112922882,
  "full_model_parameter_bytes_only": 451691528,
  "trainable_parameters": 887042,
  "adapter_parameters": 294912,
  "head_parameters": 592130,
  "trainable_parameter_bytes": 3548168,
  "all_parameter_bytes": 455239696,
  "train_seconds": 2.3013311000031536,
  "proxy_accuracy_before": 0.5,
  "proxy_accuracy_after": 0.5,
  "eval_predictions": [
    0,
    0,
    0,
    0
  ],
  "full_finetuning_training": "미측정",
  "peak_memory": "미측정"
}

설정 | 학습 파라미터 | proxy 정확도 전→후 | 학습 초 | 파라미터 MiB
r=4 | 739,586 | 0.50→0.50 | 2.24 | 433.59
r=8 | 887,042 | 0.50→0.50 | 2.30 | 434.15
전체 미세조정: 파라미터 count만 계산, 학습 시간·정확도 미측정
실행 결과: week03-results.json


## 6. 결과 해석과 과제 1

1. r = 4와 비교 rank에서 어댑터 파라미터는 얼마나 늘었는가? 분류 head까지 합친 학습 파라미터가 정확히 두 배가 아닌 이유 설명
2. proxy 평가 4개에서 한 건 차이는 정확도 몇 차이인가? 이 작은 숫자를 제주 전체 문서 분류 성능으로 일반화할 수 없는 이유 설명
3. 각 설정의 실제 학습 시간·파라미터 저장 bytes 기록. peak 메모리와 전체 미세조정의 학습 성능은 미측정으로 보존
4. alpha/r을 유지한 이유와, 데이터·step·seed·배치를 고정한 이유 설명
5. 제공된 비교표에 자신의 결과와 단위를 기록하고 도내 소규모 조직의 상정 예산 안에서 선택 근거 제시

- 데이터에 대한 대규모 성능 실험이 아니라 제공 PEFT 골격의 실제 학습·평가 흐름 확인
- 과제 1 상세: assignments/week-03/README.md
- 제출 학번 폴더에는 이름·연락처 미포함. 공개 원문 전체와 다운로드 캐시는 커밋하지 않는 원칙


## 과제 1 답안

### 1. 가정한 메모리·시간 예산

- 도내 소규모 조직 가정: 전용 GPU 없음, Colab 무료 CPU 정도의 환경(CPU 2스레드)
- 시간: 데이터 준비부터 학습·평가까지 30분 이내
- 메모리: 학습에 쓰는 메모리 2GB 이내
- 실제 실행: 로컬 PC CPU, torch 스레드 2, float32 (Colab 무료 CPU와 비슷하게 맞춤)

### 2. 비교표

| 기법 | 학습 파라미터 수 | 전체 대비 비율 | GPU/런타임 | 학습 시간 | 평가 정확도 (proxy, 4건) | 메모리 사용량 |
|---|---|---|---|---|---|---|
| 전체 미세조정 | 112,922,882 (기준선) | 100% | 미실행 | 미측정 | 미측정 | 파라미터 430.8 MiB (학습 시 peak 미측정) |
| LoRA r=4 (내 랭크 값) | 739,586 | 0.65% | CPU 2스레드, GPU 없음 | train 2.24초 (셀 전체 5.58초) | 0.50 → 0.50 | 학습 파라미터 2.82 MiB / 전체 433.6 MiB, peak 미측정 |
| LoRA r=8 (랭크 바꾼 결과) | 887,042 | 0.79% | CPU 2스레드, GPU 없음 | train 2.30초 (셀 전체 4.18초) | 0.50 → 0.50 | 학습 파라미터 3.38 MiB / 전체 434.2 MiB, peak 미측정 |

- 학습 파라미터 수: `requires_grad=True`인 파라미터 합계(LoRA 어댑터 + 분류 head). `print_trainable_parameters()`의 trainable params와 같은 기준
  - r=4: 어댑터 147,456 + head 592,130
  - r=8: 어댑터 294,912 + head 592,130
- 전체 대비 비율: 원래 모델 112,922,882개 기준
- 학습 시간: `trainer.train()` 구간(6 step, 배치 2). 괄호는 모델 로딩·평가까지 포함한 셀 실행 시간
- 평가 정확도: 키워드 규칙으로 만든 proxy label을 맞춘 비율(평가 4건). 감성 분석 성능 아님
- 메모리: 파라미터 저장 bytes만 측정. 활성값·gradient·optimizer 상태를 포함한 peak 메모리는 미측정
- 원본 결과: `week03-results.json`

### 3. 예산 안에서 선택한 기법과 근거

- 선택: LoRA r=4
- 전체 미세조정이 아니라 PEFT를 고른 이유
  - 전체 미세조정은 1억 1천만 개 파라미터를 모두 학습함. Adam 기준으로 가중치+gradient+optimizer 상태만 대략 파라미터당 16바이트라서 112,922,882 × 16 ≈ 1.7GB(추정, 실측 아님). 여기에 활성값까지 더하면 2GB 예산을 넘을 가능성이 큼
  - LoRA는 학습 파라미터가 0.65%(약 74만 개)라 gradient·optimizer 상태가 수 MB 수준. 본체 가중치(약 431 MiB)는 고정이라 그대로 읽기만 함
  - 시간도 CPU에서 6 step 학습이 2초대라 30분 예산 안에서 step이나 데이터를 늘릴 여유가 있음
  - 저장할 때도 어댑터+head만 저장하면 돼서(약 3MB) 용도별로 여러 개를 두기 쉬움
- r=4를 고른 이유: 이번 실험에서는 r=8로 올려도 정확도 차이가 없었고, 같은 결과면 파라미터가 적은 쪽이 예산상 유리함
- 한계: 전체 미세조정은 실제로 돌리지 않아서 정확도·시간 비교는 못 했고, 위 메모리는 추정치임

### 4. 랭크 변경 시 달라진 점

- 어댑터 파라미터는 147,456 → 294,912로 정확히 2배. 전체 학습 파라미터는 739,586 → 887,042로 약 1.2배
- 학습 시간은 2.24초 → 2.30초로 거의 같음(어댑터가 전체 연산에서 차지하는 비중이 작음)
- 정확도는 둘 다 0.50 → 0.50으로 변화 없음. 두 설정 모두 평가 4건을 전부 0으로 예측했음
- 평가 loss는 둘 다 0.689 → 0.797 정도로 오히려 올라감. 6 step, 학습 데이터 8개로는 규칙을 배우기에 부족해서 한쪽 클래스로 쏠린 것 같음. 그래서 이번 결과로는 랭크가 성능에 주는 영향을 판단할 수 없음

### 노트북 6번 질문 답

1. 어댑터는 query·value 12층 × 2곳에 r×768 + 768×r씩 붙어서 r에 정확히 비례함(147,456 → 294,912). 그런데 분류 head(592,130개)는 랭크와 상관없이 고정이라 합친 학습 파라미터는 2배가 아니라 1.2배만 늘어남
2. 평가가 4건이라 한 건 차이는 0.25. 문단 4개, 그것도 키워드 규칙 label이라 제주 문서 전체 분류 성능으로 일반화할 수 없음. 이번 0.50도 전부 0으로 찍어서 label 0인 2건만 맞은 것이라 우연 수준과 같음
3. 학습 시간: r=4 2.24초, r=8 2.30초. 파라미터 저장 bytes: r=4 학습 2,958,344 / 전체 454,649,872, r=8 학습 3,548,168 / 전체 455,239,696. peak 메모리와 전체 미세조정 학습 성능은 미측정
4. alpha/r을 2로 고정해서 LoRA 업데이트 크기 배율을 같게 맞춤. 그래야 rank만 바뀐 효과를 볼 수 있음. 데이터·step·seed·배치도 고정해야 결과 차이를 rank 때문이라고 말할 수 있음(실제로 학습 전 평가 loss가 두 설정에서 똑같이 0.6895로 나옴)
5. 위 비교표와 3번 근거 참고
